## Librerías

In [36]:
from __future__ import annotations

import re
import time as tm
from datetime import date, timedelta, time as dtime
import pandas as pd
import zipfile
import shutil
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    TimeoutException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
)
from typing import Optional, Tuple, Dict, List, Set, Any
from openpyxl import load_workbook
import mysql.connector



## Datos 

### No Modificar

In [37]:
# --------- Operación Real ---------
MESES = {
    1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"
}

MASHUP_URL = "https://qap-prd.coordinador.cl/ext/extensions/mashup_Generacion_Real_Descargable/mashup_Generacion_Real_Descargable.html"

DOWNLOAD_DIR = Path("/Users/valentinauretadelafuente/Desktop/Operacion Real/Datos Operación")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# --------- Programación ---------
URL_DOCS = "https://programa.coordinador.cl/operacion/pcp/bases-modelo"

# --------- Base de datos ---------
MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "rootvale123"
MYSQL_DB = "costos_variables"

MYSQL = dict(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="rootvale123",
    database="costos_variables",
)

conn = mysql.connector.connect(
    host="127.0.0.1", port=3306, user="root", password="rootvale123",
    database="costos_variables"
)

# --------- Barras ---------
DB_OR   = "costos_variables"
DB_OUT  = "planificacion"

TABLE_OR  = f"{DB_OR}.costos_marginales"   
TABLE_OUT = f"{DB_OUT}.barras"  

BARRAS_MAP = {
    "A.JAHUEL":      "AJahuel220",
    "CRUCERO":       "Crucero220",
    "MAITENCILLO":   "Maitencillo220",
    "P.MONTT":       "PMontt220",
    "QUILLOTA":      "Quillota220",
    "CHARRUA":       "Charrua220",
    "D.ALMAGRO":     "DAlmagro220",
}

# --------- Reservas ---------
DB_DEST = "planificacion"
TABLE_DEST = "reserva_conf"

CENTRALES_CARBON = [
    "ANGAMOS-ANG1_CAR + BESS",
    "ANGAMOS-ANG2_CAR + BESS",
    "COCHRANE-CCH1_CAR + BESS",
    "COCHRANE-CCH2_CAR + BESS",
]
T_SRC = "planificacion.reserva_conf"     # origen (normalizada)
DB_OUT = "planificacion"
T_OUT = "reservas_tec"                  # destino: planificacion.reservas_tec

TIPO2PREF = {
    "Primaria": "CPF",
    "Secundaria": "CSF",
    "Terciaria": "CTF",
}

KEY_COLS = ["Fecha", "hora"]

### Modificar

In [38]:
SELECCIONAR_TODOS_ANIOS = False
ANIOS = [2026]                          # Modificar año

SELECCIONAR_TODOS_MESES = False
MESES_SELECCION = [MESES[3]]            # Modificar mes

FINAL_NAME = f"Generacion_real_{ANIOS[0]}_{MESES_SELECCION[0]}.csv"

ini = date(2026, 3, 16)
fin = date(2026, 3, 28)

## Funciones

In [39]:
def seleccionar_valores(driver, wait, valores=None, seleccionar_todos=False):
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "[data-testid='listbox.item'] [role='row']")
    ))

    rows = driver.find_elements(By.CSS_SELECTOR, "[data-testid='listbox.item'] [role='row']")

    if seleccionar_todos:
        for row in rows:
            if row.get_attribute("aria-selected") != "true":
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", row)
                row.click()
                tm.sleep(0.1)
        return

    if not valores:
        return

    for val in valores:
        val_text = str(val)
        row = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f"//div[@role='row' and .//span[normalize-space()='{val_text}']]")
        ))
        if row.get_attribute("aria-selected") != "true":
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", row)
            row.click()
            tm.sleep(0.1)

def confirmar_seleccion(driver, wait, titulo):
    btn = wait.until(EC.element_to_be_clickable(
        (By.XPATH, f"//h6[@title='{titulo}']/ancestor::div[contains(@class,'css-kbfjxp')][1]"
                   f"//button[@data-testid='actions-toolbar-confirm']")
    ))
    driver.execute_script("arguments[0].click();", btn)
    tm.sleep(0.5)

def click_descargar(driver, wait):
    btn = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//button[.//span[normalize-space()='Descargar'] or contains(., 'Descargar') or .//*[contains(@class,'lui-icon--download')]]"
                   " | //a[.//span[normalize-space()='Descargar'] or contains(., 'Descargar') or .//*[contains(@class,'lui-icon--download')]]")
    ))
    driver.execute_script("arguments[0].click();", btn)
    tm.sleep(1)

def esperar_descarga_finalizada(download_dir: Path, timeout: int = 180) -> Path:
    end = tm.time() + timeout
    before = {p.name for p in download_dir.glob("*")}

    while tm.time() < end:
        current = list(download_dir.glob("*"))
        cr = list(download_dir.glob("*.crdownload"))

        nuevos = [p for p in current if p.name not in before and p.suffix.lower() == ".csv"]
        if nuevos and not cr:
            # devuelve el más nuevo
            return max(nuevos, key=lambda p: p.stat().st_mtime)

        tm.sleep(0.5)

    raise TimeoutError(f"No se detectó descarga en {timeout}s en {download_dir}")

def mover_y_renombrar(archivo_descargado: Path, destino_dir: Path, final_name: str) -> Path:
    destino = destino_dir / final_name
    # Si ya existe, lo reemplaza
    if destino.exists():
        destino.unlink()
    archivo_descargado.rename(destino)
    return destino

def asegurar_carpetas(base_dir: Path) -> Tuple[Path, Path, Path, Path, Path]:
    programas_dir = base_dir / "Programas"
    programas_prg = programas_dir / "PRG"
    programas_po = programas_dir / "PO"
    tco_dir = base_dir / "TCO"
    zips_dir = base_dir / "ZIPS"  # opcional: guardar zips descargados (para evitar redescargar)
    tmp_download = base_dir / "_tmp_download"

    programas_prg.mkdir(parents=True, exist_ok=True)
    programas_po.mkdir(parents=True, exist_ok=True)
    tco_dir.mkdir(parents=True, exist_ok=True)
    zips_dir.mkdir(parents=True, exist_ok=True)
    tmp_download.mkdir(parents=True, exist_ok=True)

    return programas_prg, programas_po, tco_dir, zips_dir, tmp_download

def limpiar_tmp(tmp: Path, patterns: List[str]):
    for pat in patterns:
        for p in tmp.glob(pat):
            try:
                if p.is_file():
                    p.unlink()
            except Exception:
                pass

def esperar_archivo_sin_crdownload(
    carpeta: Path,
    nombre_objetivo: str,
    timeout_s: int = 240,
    poll_s: float = 1.0,
) -> Optional[Path]:
    destino = carpeta / nombre_objetivo
    crdownload = carpeta / f"{nombre_objetivo}.crdownload"

    t0 = tm.time()
    while tm.time() - t0 < timeout_s:
        if destino.exists() and not crdownload.exists():
            return destino
        tm.sleep(poll_s)

    return None

def make_driver(download_dir: Path) -> webdriver.Chrome:
    chrome_options = Options()
    chrome_options.add_argument("--start-minimized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--safebrowsing-disable-download-protection")

    prefs = {
        "download.default_directory": str(download_dir.resolve()),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
        "profile.default_content_setting_values.automatic_downloads": 1,
    }
    chrome_options.add_experimental_option("prefs", prefs)

    return webdriver.Chrome(options=chrome_options)

def _kill_overlays(driver: webdriver.Chrome):
    driver.execute_script(
        """
      const killers = Array.from(document.querySelectorAll('div,section,aside,dialog'))
        .filter(el => {
          const s = getComputedStyle(el);
          const z = parseInt(s.zIndex || '0');
          const big = (el.offsetHeight > 80 && el.offsetWidth > 80);
          const fixed = (s.position === 'fixed' || s.position === 'sticky');
          return big && fixed && z >= 1000;
        });
      killers.slice(0,10).forEach(el => el.style.display='none');
    """
    )

def _try_close_common_popups(driver: webdriver.Chrome):
    close_xpaths = [
        "//button[contains(.,'Aceptar')] | //button[contains(.,'Entendido')] | //button[contains(.,'OK')]",
        "//button[contains(.,'Continuar')] | //a[contains(.,'Continuar')]",
        "//*[@aria-label='Cerrar' or @aria-label='Close']",
        "//button[contains(@class,'close') or contains(@class,'btn-close')]",
        "//button[contains(.,'Cerrar')]",
    ]
    for xp in close_xpaths:
        try:
            el = WebDriverWait(driver, 1).until(EC.element_to_be_clickable((By.XPATH, xp)))
            driver.execute_script("arguments[0].click();", el)
            tm.sleep(0.2)
        except Exception:
            pass

def set_resultados_por_pagina(driver: webdriver.Chrome, wait: WebDriverWait, n: int = 30):
    _try_close_common_popups(driver)
    _kill_overlays(driver)

    select_el = None
    candidates = driver.find_elements(By.XPATH, "//*[contains(.,'Resultados por página')]/following::select[1]")
    if candidates:
        select_el = candidates[0]
    else:
        sels = driver.find_elements(By.TAG_NAME, "select")
        if sels:
            select_el = sels[-1]

    if select_el is None:
        raise RuntimeError("No se encontró el selector 'Resultados por página'.")

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", select_el)
    tm.sleep(0.2)

    sel = Select(select_el)
    sel.select_by_visible_text(str(n))

    tm.sleep(0.6)
    wait.until(EC.presence_of_element_located((By.XPATH, "//table//tbody//tr")))

def obtener_filas_tabla(driver: webdriver.Chrome, wait: WebDriverWait):
    wait.until(EC.presence_of_element_located((By.XPATH, "//table//tbody")))
    return driver.find_elements(By.XPATH, "//table//tbody//tr")

def extraer_nombre_archivo_de_fila(row) -> str:
    tds = row.find_elements(By.TAG_NAME, "td")
    if not tds:
        return ""
    return (tds[0].text or "").strip()

def click_descarga_en_fila(driver: webdriver.Chrome, row) -> None:
    _kill_overlays(driver)

    link = row.find_element(
        By.XPATH,
        ".//a[contains(.,'ZIP') or contains(.,'XLSX') or contains(@href,'.zip') or contains(@href,'.xlsx')]",
    )
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", link)
    tm.sleep(0.1)
    try:
        link.click()
    except (ElementClickInterceptedException, StaleElementReferenceException):
        _kill_overlays(driver)
        driver.execute_script("arguments[0].click();", link)

def ir_a_siguiente_pagina(driver: webdriver.Chrome) -> bool:
    _try_close_common_popups(driver)
    _kill_overlays(driver)

    next_xpaths = [
        # botones típicos
        "//button[@aria-label='Next' and not(@disabled)]",
        "//a[@aria-label='Next' and not(contains(@class,'disabled'))]",
        # por texto
        "//button[contains(.,'Siguiente') and not(@disabled)]",
        "//a[contains(.,'Siguiente') and not(contains(@class,'disabled'))]",
        # por ícono: botón con flecha derecha (muy común en paginación)
        "//button[.//*[name()='svg'] and not(@disabled)][last()]",
    ]

    for xp in next_xpaths:
        try:
            nxt = WebDriverWait(driver, 1).until(EC.element_to_be_clickable((By.XPATH, xp)))
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", nxt)
            tm.sleep(0.1)
            try:
                nxt.click()
            except ElementClickInterceptedException:
                _kill_overlays(driver)
                driver.execute_script("arguments[0].click();", nxt)
            tm.sleep(0.8)
            return True
        except Exception:
            continue

    return False

def subtipo_termica_por_nombre(central: str) -> Optional[str]:
    c = central.upper()
    if "SAE" in c:
        return "Bess"
    if "PFV" in c:
        return "Solar"
    if "PE" in c:
        return "Eólica"
    if "DIESEL" in c:
        return "Diésel"
    if "GNL_INFLEX" in c: 
        return "GNL INF"
    if "GLP" in c: 
        return "GLP"
    if "GNL" in c:
        return "GNL"
    if "GN" in c:
        return "GNA"
    if "HFO" in c or "IFO" in c or "FO6" in c:
        return "Fuil Oil"
    if "GEO" in c:
        return "Geotermica"
    if "CAR" in c:
        return "Carbón"
    if "BIOGAS" in c or "SANTAMARTA" in c:
        return "BioGas"
    if "ENAPBIOBIO" in c:
        return "PetCoke"
    if "BIOMASA" in c or "COGEN" in c:
        return "Biomasa"
    if "PROPANO" in c:
        return "PROPANO"
    return None

def _resolver_archivo_con_sufijos(path: str | Path) -> Path:
    path = Path(path)
    if path.exists():
        return path

    parent = path.parent
    stem = path.stem
    ext = path.suffix

    candidatos = [
        parent / f"{stem}{ext}",
        parent / f"{stem}-1{ext}",
        parent / f"{stem}-2{ext}",
    ]
    existentes = [p for p in candidatos if p.exists()]
    if existentes:
        return max(existentes, key=lambda p: p.stat().st_mtime)

    globbed = list(parent.glob(f"{stem}*{ext}"))
    if globbed:
        return max(globbed, key=lambda p: p.stat().st_mtime)

    raise FileNotFoundError(f"No se encontró: {path.name} (ni variantes -1/-2) en {parent}")

def leer_prg(prg_path: str | Path) -> Tuple[Dict, Dict, Dict, Dict]:
    prg_path = _resolver_archivo_con_sufijos(prg_path)

    tablas_energia = {
        "Hidroeléctricas de Pasada",
        "Eólicas",
        "Solares",
        "Centrales de concentración solar",
        "Térmicas",
        "Embalses y Reguladas",
    }
    wb = load_workbook(prg_path, read_only=True, data_only=True)
    ws = wb["PROGRAMA"]

    datos_por_titulo: Dict[str, Dict[str, Any]] = {}

    rows = ws.iter_rows(min_row=15, values_only=True)
    fila = next(rows, None)

    while fila is not None:
        titulo_celda = fila[2]  # Columna C
        if titulo_celda is None or str(titulo_celda).strip() == "":
            fila = next(rows, None)
            continue

        titulo = str(titulo_celda).strip()
        def norm_title(x: str) -> str:
            x = x.replace("\u00A0", " ")          # NBSP -> espacio
            x = " ".join(x.split())              # colapsa espacios
            x = x.replace("–", "-").replace("—", "-")  # guiones raros -> "-"
            return x.strip()

        titulo = norm_title(str(titulo_celda))

        es_tabla_energia = titulo in tablas_energia
        es_costos = titulo == "Costos Marginales"
        es_sistemas = titulo.startswith("Sistemas de Almacenamiento")  # incluye Carga RED/ERV
        es_reduccion = titulo == "Reducción de Renovable Estimada"

        if not (es_tabla_energia or es_costos or es_sistemas or es_reduccion):
            fila = next(rows, None)
            continue

        fila = next(rows, None)
        centrales: Dict[str, Any] = {}

        while fila is not None:
            nombre = fila[2]  # Columna C
            if nombre is None or str(nombre).strip() == "":
                break

            nombre = str(nombre).strip()
            if nombre == "Total" and titulo != "Reducción de Renovable Estimada":
                fila = next(rows, None)
                continue

            codigo = fila[3]  # Columna D
            valores = []
            for i in range(24):
                v = fila[4 + i] if (4 + i) < len(fila) else None
                valores.append(None if v is None else round(float(v), 1))

            serie = [(dtime(h, 0, 0), valores[h]) for h in range(24)]

            if es_tabla_energia:
                centrales[nombre] = (codigo, titulo, serie)
            elif es_costos:
                centrales[nombre] = (codigo, serie)
            elif es_sistemas:
                centrales[nombre] = (codigo, serie)
            elif es_reduccion:
                centrales[nombre] = serie


            fila = next(rows, None)

        datos_por_titulo[titulo] = centrales
        fila = next(rows, None)

    dict_energia: Dict[str, Any] = {}
    for k in [
        "Hidroeléctricas de Pasada",
        "Eólicas",
        "Solares",
        "Centrales de concentración solar",
        "Térmicas",
        "Embalses y Reguladas",
    ]:
        dict_energia.update(datos_por_titulo.get(k, {}))

    dict_costos = datos_por_titulo.get("Costos Marginales", {})
    dict_sistemas = datos_por_titulo.get("Sistemas de Almacenamiento", {})
    dict_reduccion = datos_por_titulo.get("Reducción de Renovable Estimada", {})

    return dict_energia, dict_costos, dict_sistemas, dict_reduccion

def leer_po_fijo(
    po_path: str | Path,
    sheet_name: str = "RESUMEN",
    first_data_row: int = 34
) -> List[Dict[str, Any]]:
    po_path = Path(po_path)
    wb = load_workbook(po_path, read_only=True, data_only=True)
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]

    def norm(x):
        if x is None:
            return ""
        return " ".join(str(x).replace("\n", " ").split()).strip()

    out: List[Dict[str, Any]] = []
    r = first_data_row

    while r <= (ws.max_row or r):
        central = norm(ws.cell(r, 2).value)  # col 2
        if central == "":
            break

        row = {
            "Central": central,
            "E_max_generable": ws.cell(r, 3).value,
            "Energia": ws.cell(r, 4).value,
            "Cmg_CVT": ws.cell(r, 5).value,
            "CVC": ws.cell(r, 6).value,
            "CVNC": ws.cell(r, 7).value,
            "CMed_Min_Tec": ws.cell(r, 8).value,
            "Combustible_Costo": ws.cell(r, 9).value,
            "Combustible_Unidad": ws.cell(r, 10).value,
        }
        out.append(row)
        r += 1

    return out

def leer_tco_cv_fijo(
    tco_path: str | Path,
    sheet_name: str = "CV",
    header_row: int = 20,
) -> List[Dict[str, Any]]:

    tco_path = Path(tco_path)
    wb = load_workbook(tco_path, read_only=True, data_only=True)
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]

    COL_CENTRAL = 1
    COL_TIPO = 2
    COL_CV = 3
    COL_EF = 4  
    COL_BARRA = 5
    COL_NOMBRE_BARRA = 6
    COL_HORAS_START = 7   
    COL_BLOQUE1 = 31      
    COL_BLOQUE2 = 32
    COL_BLOQUE3 = 33

    def norm(x: Any) -> str:
        if x is None:
            return ""
        return " ".join(str(x).replace("\n", " ").split()).strip()

    out: List[Dict[str, Any]] = []
    r = header_row + 1

    while r <= (ws.max_row or r):
        central = norm(ws.cell(r, COL_CENTRAL).value)
        if central == "":
            break

        tipo_raw = norm(ws.cell(r, COL_TIPO).value)  
        tipo = None
        subtipo = None

        if tipo_raw.upper() == "E":
            tipo = "Embalse"
        elif tipo_raw.upper() == "T":
            tipo = "Termica"
            subtipo = subtipo_termica_por_nombre(central)
        else:
            tipo = None  

        row: Dict[str, Any] = {
            "Central": central,
            "Tipo": tipo,
            "Subtipo": subtipo,
            "CV": ws.cell(r, COL_CV).value,
            "Eficiencia": ws.cell(r, COL_EF).value,
            "Barra": ws.cell(r, COL_BARRA).value,
            "Nombre_Barra": ws.cell(r, COL_NOMBRE_BARRA).value,
            "Bloque1": ws.cell(r, COL_BLOQUE1).value,
            "Bloque2": ws.cell(r, COL_BLOQUE2).value,
            "Bloque3": ws.cell(r, COL_BLOQUE3).value,
        }

        for h in range(1, 25):
            row[f"Hora{h}"] = ws.cell(r, COL_HORAS_START + (h - 1)).value

        out.append(row)
        r += 1

    return out

def migrar_esquema(conn):
    cur = conn.cursor()

    # 1) Tabla energía: cambiar UNIQUE de (Fecha, Central) -> (Fecha, Central, Subtipo)
    #    (si ya existe, se ignora el error)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS prg_energia_horaria (
      id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
      Fecha DATE NOT NULL,
      Central VARCHAR(255) NOT NULL,
      Categoria VARCHAR(80) NOT NULL,
      Subtipo VARCHAR(80) NULL,
      Hora1  DECIMAL(12,6) NULL, Hora2  DECIMAL(12,6) NULL, Hora3  DECIMAL(12,6) NULL, Hora4  DECIMAL(12,6) NULL,
      Hora5  DECIMAL(12,6) NULL, Hora6  DECIMAL(12,6) NULL, Hora7  DECIMAL(12,6) NULL, Hora8  DECIMAL(12,6) NULL,
      Hora9  DECIMAL(12,6) NULL, Hora10 DECIMAL(12,6) NULL, Hora11 DECIMAL(12,6) NULL, Hora12 DECIMAL(12,6) NULL,
      Hora13 DECIMAL(12,6) NULL, Hora14 DECIMAL(12,6) NULL, Hora15 DECIMAL(12,6) NULL, Hora16 DECIMAL(12,6) NULL,
      Hora17 DECIMAL(12,6) NULL, Hora18 DECIMAL(12,6) NULL, Hora19 DECIMAL(12,6) NULL, Hora20 DECIMAL(12,6) NULL,
      Hora21 DECIMAL(12,6) NULL, Hora22 DECIMAL(12,6) NULL, Hora23 DECIMAL(12,6) NULL, Hora24 DECIMAL(12,6) NULL,
      PRIMARY KEY (id),
      UNIQUE KEY uq_fecha_central_subtipo (Fecha, Central, Subtipo),
      KEY idx_categoria (Categoria),
      KEY idx_subtipo (Subtipo)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;
    """)

    # Si ya existía uq_fecha_central, lo eliminamos y dejamos el nuevo
    try:
        cur.execute("ALTER TABLE prg_energia_horaria DROP INDEX uq_fecha_central;")
    except mysql.connector.Error:
        pass

    # Asegurar que el nuevo índice exista (si la tabla ya existía, CREATE TABLE no lo recrea)
    try:
        cur.execute("ALTER TABLE prg_energia_horaria ADD UNIQUE KEY uq_fecha_central_subtipo (Fecha, Central, Subtipo);")
    except mysql.connector.Error:
        pass

    # 2) Tabla costos marginales
    cur.execute("""
    CREATE TABLE IF NOT EXISTS costos_marginales (
      id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
      Fecha DATE NOT NULL,
      Barra VARCHAR(255) NOT NULL,
      Codigo VARCHAR(80) NULL,
      Hora1  DECIMAL(12,6) NULL, Hora2  DECIMAL(12,6) NULL, Hora3  DECIMAL(12,6) NULL, Hora4  DECIMAL(12,6) NULL,
      Hora5  DECIMAL(12,6) NULL, Hora6  DECIMAL(12,6) NULL, Hora7  DECIMAL(12,6) NULL, Hora8  DECIMAL(12,6) NULL,
      Hora9  DECIMAL(12,6) NULL, Hora10 DECIMAL(12,6) NULL, Hora11 DECIMAL(12,6) NULL, Hora12 DECIMAL(12,6) NULL,
      Hora13 DECIMAL(12,6) NULL, Hora14 DECIMAL(12,6) NULL, Hora15 DECIMAL(12,6) NULL, Hora16 DECIMAL(12,6) NULL,
      Hora17 DECIMAL(12,6) NULL, Hora18 DECIMAL(12,6) NULL, Hora19 DECIMAL(12,6) NULL, Hora20 DECIMAL(12,6) NULL,
      Hora21 DECIMAL(12,6) NULL, Hora22 DECIMAL(12,6) NULL, Hora23 DECIMAL(12,6) NULL, Hora24 DECIMAL(12,6) NULL,
      PRIMARY KEY (id),
      UNIQUE KEY uq_fecha_barra (Fecha, Barra),
      KEY idx_barra (Barra)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;
    """)

    conn.commit()
    cur.close()

def serie_a_24(serie):
    vals = [v for (_t, v) in serie]
    if len(vals) != 24:
        vals = (vals + [None] * 24)[:24]
    return vals

def q_ident(s: str) -> str:
    return "`" + s.replace("`", "") + "`"

def q_literal(s: str) -> str:
    return s.replace("'", "''")

def ensure_out_table(cur):
    cols = ",\n  ".join([f"{q_ident(k)} DECIMAL(18,6) NULL" for k in BARRAS_MAP.keys()])
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_OUT}`;")
    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_OUT} (
      Fecha DATE NOT NULL,
      hora  TINYINT NOT NULL,
      {cols},
      PRIMARY KEY (Fecha, hora)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;
    """)
    # por si agregas barras nuevas después
    for k in BARRAS_MAP.keys():
        try:
            cur.execute(f"ALTER TABLE {TABLE_OUT} ADD COLUMN {q_ident(k)} DECIMAL(18,6) NULL;")
        except mysql.connector.Error:
            pass

def upsert_barras_from_wide_hours(fecha_inicio, fecha_fin):
    conn = mysql.connector.connect(
    host="127.0.0.1", port=3306, user="root", password="rootvale123",
    database="costos_variables")
    try:
        cur = conn.cursor()
        ensure_out_table(cur)

        # 1) UNPIVOT horas: Hora1..Hora24 -> filas (Fecha,hora,Barra,valor)
        #    Hora1 -> hora=0, ..., Hora24 -> hora=23
        unpivot_parts = []
        for h in range(1, 25):
            hora_out = h - 1
            unpivot_parts.append(f"""
            SELECT
              o.Fecha,
              {hora_out} AS hora,
              o.Barra,
              o.{q_ident(f"Hora{h}")} AS valor
            FROM {TABLE_OR} o
            WHERE o.Fecha BETWEEN %s AND %s
              AND o.Barra IN ({", ".join(["%s"]*len(BARRAS_MAP))})
            """.strip())
        unpivot_sql = "\nUNION ALL\n".join(unpivot_parts)

        # 2) PIVOT a columnas por barra destino
        cols_insert = ["Fecha", "hora"] + [q_ident(k) for k in BARRAS_MAP.keys()]
        sums = []
        updates = []

        for col_out, barra_src in BARRAS_MAP.items():
            sums.append(
                f"MAX(CASE WHEN u.Barra = '{q_literal(barra_src)}' THEN u.valor ELSE NULL END) AS {q_ident(col_out)}"
            )
            updates.append(f"{q_ident(col_out)}=VALUES({q_ident(col_out)})")

        sql = f"""
        INSERT INTO {TABLE_OUT} ({", ".join(cols_insert)})
        SELECT
          u.Fecha,
          u.hora,
          {", ".join(sums)}
        FROM (
          {unpivot_sql}
        ) u
        GROUP BY u.Fecha, u.hora
        ON DUPLICATE KEY UPDATE
          {", ".join(updates)};
        """

        # parámetros: por cada SELECT del union => (fecha_ini, fecha_fin, *barras)
        barras_params = list(BARRAS_MAP.values())
        params = []
        for _ in range(24):
            params.extend([fecha_inicio, fecha_fin, *barras_params])

        cur.execute(sql, tuple(params))
        conn.commit()

        cur.execute(f"SELECT COUNT(*) FROM {TABLE_OUT} WHERE Fecha BETWEEN %s AND %s;", (fecha_inicio, fecha_fin))
        print("Filas en planificacion.barras (rango):", cur.fetchone()[0])

    finally:
        conn.close()

def _to_float_or_none(x):
    if x is None:
        return None
    try:
        if isinstance(x, str):
            s = x.strip()
            if s.count(",") == 1 and s.count(".") >= 1:
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", ".")
            return float(s)
        return float(x)
    except Exception:
        return None

def upsert_po_termicas(conn, fecha, po_rows):
    sql = """
    INSERT INTO costos_variables.po
    (Fecha, Central, Categoria, Subtipo,
     E_max_generable, Energia, Cmg_CVT, CVC, CVNC, CMed_Min_Tec,
     Combustible_Costo, Combustible_Unidad)
    VALUES
    (%s, %s, %s, %s,
     %s, %s, %s, %s, %s, %s,
     %s, %s)
    ON DUPLICATE KEY UPDATE
      Categoria=VALUES(Categoria),
      Subtipo=VALUES(Subtipo),
      E_max_generable=VALUES(E_max_generable),
      Energia=VALUES(Energia),
      Cmg_CVT=VALUES(Cmg_CVT),
      CVC=VALUES(CVC),
      CVNC=VALUES(CVNC),
      CMed_Min_Tec=VALUES(CMed_Min_Tec),
      Combustible_Costo=VALUES(Combustible_Costo),
      Combustible_Unidad=VALUES(Combustible_Unidad);
    """

    params_list = []
    for r in po_rows:
        central = r.get("Central")
        if central is None or str(central).strip() == "":
            continue

        central_txt = str(central).strip()
        categoria = "Termicas"
        subtipo = subtipo_termica_por_nombre(central_txt)

        params_list.append((
            fecha,
            central_txt,
            categoria,
            subtipo,
            _to_float_or_none(r.get("E_max_generable")),
            _to_float_or_none(r.get("Energia")),
            _to_float_or_none(r.get("Cmg_CVT")),
            _to_float_or_none(r.get("CVC")),
            _to_float_or_none(r.get("CVNC")),
            _to_float_or_none(r.get("CMed_Min_Tec")),
            _to_float_or_none(r.get("Combustible_Costo")),
            None if r.get("Combustible_Unidad") in (None, "") else str(r.get("Combustible_Unidad")).strip(),
        ))

    if not params_list:
        return 0

    cur = conn.cursor()
    cur.executemany(sql, params_list)  
    conn.commit()                      
    cur.close()
    return len(params_list)

def _to_float_or_none(x):
    if x is None:
        return None
    try:
        if isinstance(x, str):
            s = x.strip()
            if s.count(",") == 1 and s.count(".") >= 1:
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", ".")
            return float(s)
        return float(x)
    except Exception:
        return None

def upsert_tco(conn, fecha: date, rows: list[dict]):
    sql = """
    INSERT INTO costos_variables.tco
    (Fecha, Central, Tipo, Subtipo, CV, Eficiencia, Barra, Nombre_Barra,
     Hora1, Hora2, Hora3, Hora4, Hora5, Hora6, Hora7, Hora8, Hora9, Hora10, Hora11, Hora12,
     Hora13, Hora14, Hora15, Hora16, Hora17, Hora18, Hora19, Hora20, Hora21, Hora22, Hora23, Hora24,
     Bloque1, Bloque2, Bloque3)
    VALUES
    (%s, %s, %s, %s, %s, %s, %s, %s,
     %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
     %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
     %s, %s, %s)
    ON DUPLICATE KEY UPDATE
      Tipo=VALUES(Tipo),
      Subtipo=VALUES(Subtipo),
      CV=VALUES(CV),
      Eficiencia=VALUES(Eficiencia),
      Barra=VALUES(Barra),
      Nombre_Barra=VALUES(Nombre_Barra),
      Hora1=VALUES(Hora1), Hora2=VALUES(Hora2), Hora3=VALUES(Hora3), Hora4=VALUES(Hora4),
      Hora5=VALUES(Hora5), Hora6=VALUES(Hora6), Hora7=VALUES(Hora7), Hora8=VALUES(Hora8),
      Hora9=VALUES(Hora9), Hora10=VALUES(Hora10), Hora11=VALUES(Hora11), Hora12=VALUES(Hora12),
      Hora13=VALUES(Hora13), Hora14=VALUES(Hora14), Hora15=VALUES(Hora15), Hora16=VALUES(Hora16),
      Hora17=VALUES(Hora17), Hora18=VALUES(Hora18), Hora19=VALUES(Hora19), Hora20=VALUES(Hora20),
      Hora21=VALUES(Hora21), Hora22=VALUES(Hora22), Hora23=VALUES(Hora23), Hora24=VALUES(Hora24),
      Bloque1=VALUES(Bloque1), Bloque2=VALUES(Bloque2), Bloque3=VALUES(Bloque3);
    """

    params_list = []
    for r in rows:
        params_list.append((
            fecha,
            r["Central"],
            r.get("Tipo"),
            r.get("Subtipo"),
            _to_float_or_none(r.get("CV")),
            _to_float_or_none(r.get("Eficiencia")),
            None if r.get("Barra") is None else str(r.get("Barra")).strip(),
            None if r.get("Nombre_Barra") is None else str(r.get("Nombre_Barra")).strip(),
            *[_to_float_or_none(r.get(f"Hora{i}")) for i in range(1, 25)],
            _to_float_or_none(r.get("Bloque1")),
            _to_float_or_none(r.get("Bloque2")),
            _to_float_or_none(r.get("Bloque3")),
        ))

    cur = conn.cursor()
    cur.executemany(sql, params_list)
    conn.commit()          
    cur.close()

def leer_tco_cv_fijo_rapido(
    tco_path: str | Path,
    sheet_name: str = "CV",
    header_row: int = 20,
) -> List[Dict[str, Any]]:

    tco_path = Path(tco_path)

    wb = load_workbook(
        tco_path,
        read_only=True,
        data_only=True,
        keep_vba=False,   # importante: no necesitas macros para leer datos
        keep_links=False  # evita cargar vínculos
    )
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]

    def norm(x: Any) -> str:
        if x is None:
            return ""
        return " ".join(str(x).replace("\n", " ").split()).strip()

    out: List[Dict[str, Any]] = []

    for row in ws.iter_rows(
        min_row=header_row + 1,
        min_col=1,
        max_col=33,
        values_only=True
    ):
        central = norm(row[0])
        if central == "":
            break

        tipo_raw = norm(row[1]).upper()  # E/T
        if tipo_raw == "E":
            tipo = "Embalse"
            subtipo = None
        elif tipo_raw == "T":
            tipo = "Termica"
            subtipo = subtipo_termica_por_nombre(central)
        else:
            tipo = None
            subtipo = None

        d: Dict[str, Any] = {
            "Central": central,
            "Tipo": tipo,
            "Subtipo": subtipo,
            "CV": row[2],
            "Eficiencia": row[3],
            "Barra": row[4],
            "Nombre_Barra": row[5],
            "Bloque1": row[30],
            "Bloque2": row[31],
            "Bloque3": row[32],
        }

        for i in range(24):
            d[f"Hora{i+1}"] = row[6 + i]

        out.append(d)

    wb.close()
    return out

def actualizar_subtipos_tco():
    conn = mysql.connector.connect(**MYSQL)
    try:
        cur = conn.cursor()

        reglas = [
            ("BESS", "Central LIKE 'BAT%'"),
            ("Diésel", "Central LIKE '%DIE'"),
            ("CS", "Central LIKE '%CS'"),
        ]

        total_actualizadas = 0

        for subtipo_nuevo, condicion in reglas:
            cur.execute(f"""
                SELECT COUNT(*)
                FROM costos_variables.tco
                WHERE {condicion}
                  AND (Subtipo IS NULL OR Subtipo <> %s)
            """, (subtipo_nuevo,))
            n = cur.fetchone()[0]
            print(f"Filas a actualizar a {subtipo_nuevo}: {n}")

            cur.execute(f"""
                UPDATE costos_variables.tco
                SET Subtipo = %s
                WHERE {condicion}
                  AND (Subtipo IS NULL OR Subtipo <> %s)
            """, (subtipo_nuevo, subtipo_nuevo))
            conn.commit()

            print(f"Filas actualizadas a {subtipo_nuevo}: {cur.rowcount}")
            total_actualizadas += cur.rowcount

        print(f"Total filas actualizadas: {total_actualizadas}")

    finally:
        conn.close()

def _norm(x) -> str:
    if x is None:
        return ""
    s = str(x).replace("\u00A0", " ")
    s = " ".join(s.split())
    return s.strip()

def fecha_desde_nombre_prg(prg_path: str | Path) -> date:
    """
    Extrae YYMMDD desde el nombre (ej: PRG250101.xlsx -> 2025-01-01).
    Toma los últimos 6 dígitos consecutivos del stem.
    """
    p = Path(prg_path)
    stem = p.stem  # PRG250101 o PRG250101-1
    digits = "".join(ch for ch in stem if ch.isdigit())
    if len(digits) < 6:
        raise ValueError(f"No pude extraer YYMMDD desde el nombre: {p.name}")
    yymmdd = digits[-6:]
    yy = int(yymmdd[0:2])
    mm = int(yymmdd[2:4])
    dd = int(yymmdd[4:6])
    return date(2000 + yy, mm, dd)

def _find_in_rows(
    rows: List[Tuple],
    needle: str,
    *,
    min_r: int,
    min_c: int,
    max_r: int,
    max_c: int
) -> Optional[Tuple[int, int]]:
    needle_n = _norm(needle).lower()
    R = min(len(rows), max_r)
    for r in range(min_r, R):
        row = rows[r]
        C = min(len(row), max_c)
        for c in range(min_c, C):
            if _norm(row[c]).lower() == needle_n:
                return r, c
    return None

def leer_conf_rapido(
    prg_path: str | Path,
    sheet_name: str,
    tipo_reserva: str,
    *,
    start_row: int = 6,   # fila excel (1-indexed) donde está "CONFIGURACIÓN"
    start_col: int = 7,   # col excel (1-indexed) G
    n_horas: int = 24
) -> List[Tuple[date, int, str, str, float, float]]:


    prg_path = _resolver_archivo_con_sufijos(prg_path)
    fecha = fecha_desde_nombre_prg(prg_path)

    wb = load_workbook(prg_path, read_only=True, data_only=True)
    if sheet_name not in wb.sheetnames:
        raise ValueError(f"{prg_path.name} no tiene hoja '{sheet_name}'. Hojas: {wb.sheetnames}")
    ws = wb[sheet_name]

    # Leer solo un área razonable para velocidad.
    # Ajusta si alguna hoja viene más grande.
    max_row = min(ws.max_row, 600)
    max_col = min(ws.max_column, 220)

    rows = list(ws.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col, values_only=True))

    # Convertir start_row/start_col a 0-index
    sr = max(start_row - 1, 0)
    sc = max(start_col - 1, 0)

    # Encontrar CONFIGURACIÓN cerca del header
    pos_cfg = _find_in_rows(rows, "CONFIGURACIÓN", min_r=sr, min_c=sc, max_r=sr + 30, max_c=sc + 30)
    if pos_cfg:
        header_r, cfg_c = pos_cfg
    else:
        header_r, cfg_c = sr, sc  # fallback: asumir (fila 6, col G)

    # Encontrar SUBIDA / BAJADA (buscamos en las primeras ~80 filas desde col G)
    pos_sub = _find_in_rows(rows, "SUBIDA", min_r=0, min_c=sc, max_r=min(80, len(rows)), max_c=max_col)
    pos_baj = _find_in_rows(rows, "BAJADA", min_r=0, min_c=sc, max_r=min(80, len(rows)), max_c=max_col)
    if not pos_sub or not pos_baj:
        raise ValueError(f"No encontré 'SUBIDA' y/o 'BAJADA' en '{sheet_name}' (desde col {start_col}).")

    sub_r, sub_c0 = pos_sub
    baj_r, baj_c0 = pos_baj

    # Fila de horas suele ser 1 debajo del título
    horas_row_sub = sub_r + 1
    horas_row_baj = baj_r + 1

    def build_hour_map(row_idx: int, col_start_idx: int) -> Dict[int, int]:
        m: Dict[int, int] = {}
        if row_idx >= len(rows):
            return {h: col_start_idx + (h - 1) for h in range(1, n_horas + 1)}
        row = rows[row_idx]
        for j in range(n_horas):
            c = col_start_idx + j
            if c >= len(row):
                continue
            v = row[c]
            try:
                h = int(str(v).strip())
            except Exception:
                continue
            if 1 <= h <= n_horas:
                m[h] = c
        if len(m) < n_horas:
            m = {h: col_start_idx + (h - 1) for h in range(1, n_horas + 1)}
        return m

    sub_hmap = build_hour_map(horas_row_sub, sub_c0)
    baj_hmap = build_hour_map(horas_row_baj, baj_c0)

    def to_float(x) -> float:
        if x is None or _norm(x) == "":
            return 0.0
        try:
            return float(x)
        except Exception:
            return 0.0

    out: List[Tuple[date, int, str, str, float, float]] = []

    data_start = header_r + 1  # datos bajo "CONFIGURACIÓN"
    empty_streak = 0

    for r in range(data_start, len(rows)):
        row = rows[r]
        cfg = _norm(row[cfg_c] if cfg_c < len(row) else None)

        if not cfg:
            empty_streak += 1
            if empty_streak >= 5:
                break
            continue
        empty_streak = 0

        for h in range(1, n_horas + 1):
            cs = sub_hmap[h]
            cb = baj_hmap[h]
            v_sub = row[cs] if cs < len(row) else None
            v_baj = row[cb] if cb < len(row) else None
            out.append((fecha, h - 1, cfg, tipo_reserva, to_float(v_sub), to_float(v_baj)))

    return out

def ensure_table_reserva_conf(conn):
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_DEST}`;")
    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS `{DB_DEST}`.`{TABLE_DEST}` (
      Fecha        DATE NOT NULL,
      hora         TINYINT NOT NULL,         -- 0..23
      Central VARCHAR(255) NOT NULL,
      TipoReserva  VARCHAR(20) NOT NULL,     -- Primaria/Secundaria/Terciaria
      SUBIDA       DECIMAL(18,6) NULL,
      BAJADA       DECIMAL(18,6) NULL,
      PRIMARY KEY (Fecha, hora, Configuracion, TipoReserva)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)
    conn.commit()

def upsert_reserva_conf(prg_path: str | Path):
    prg_path = _resolver_archivo_con_sufijos(prg_path)

    targets = [
        ("Reservas CPF - Conf", "Primaria", 6, 7),
        ("Reservas CSF - Conf", "Secundaria",6, 2),
        ("Reservas CTF - Conf", "Terciaria",6, 2),
    ]

    all_rows: List[Tuple[date, int, str, str, float, float]] = []
    for sheet, tipo, sr, sc in targets:
        all_rows.extend(
            leer_conf_rapido(
                prg_path,
                sheet,
                tipo,
                start_row=sr,
                start_col=sc,
                n_horas=24
            )
        )
    if not all_rows:
        print("Sin filas para insertar.")
        return

    conn = mysql.connector.connect(**MYSQL)
    try:
        ensure_table_reserva_conf(conn)
        cur = conn.cursor()

        sql = f"""
        INSERT INTO `{DB_DEST}`.`{TABLE_DEST}`
          (Fecha, hora, Central, TipoReserva, SUBIDA, BAJADA)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
          SUBIDA = VALUES(SUBIDA),
          BAJADA = VALUES(BAJADA);
        """
        cur.executemany(sql, all_rows)
        conn.commit()

        print(f"OK: upsert {len(all_rows)} filas en {DB_DEST}.{TABLE_DEST} ({Path(prg_path).name})")
    finally:
        conn.close()

def _daterange(d1: date, d2: date):
    d = d1
    while d <= d2:
        yield d
        d += timedelta(days=1)

def upsert_reserva_conf_rango(
    fecha_inicio: date,
    fecha_fin: date,
    carpeta_prg: str | Path = "Programas/PRG",
    patron: str = "PRG{yy}{mm}{dd}.xlsx",
):

    carpeta_prg = Path(carpeta_prg)

    ok = 0
    no_file = 0
    errors = 0

    for f in _daterange(fecha_inicio, fecha_fin):
        base = patron.format(yy=f.strftime("%y"), mm=f.strftime("%m"), dd=f.strftime("%d"))
        path = carpeta_prg / base
        try:
            upsert_reserva_conf(path)
            ok += 1
        except FileNotFoundError:
            print(f"[NO FILE] {f} -> {path.name}")
            no_file += 1
        except Exception as e:
            print(f"[ERROR] {f} -> {path.name}: {e}")
            errors += 1

    print(f"Resumen: OK={ok}, NO_FILE={no_file}, ERROR={errors}")

def column_exists(cur, schema: str, table: str, column: str) -> bool:
    cur.execute("""
        SELECT COUNT(*)
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA=%s AND TABLE_NAME=%s AND COLUMN_NAME=%s
    """, (schema, table, column))
    return cur.fetchone()[0] > 0

def add_and_fill_subtipo():
    conn = mysql.connector.connect(**MYSQL)
    try:
        cur = conn.cursor()

        # 1) Crear columna Subtipo si no existe
        if not column_exists(cur, "planificacion", "reserva_conf", "Subtipo"):
            cur.execute("""
                ALTER TABLE planificacion.reserva_conf
                ADD COLUMN Subtipo VARCHAR(100) NULL
            """)
            conn.commit()

        # 2) Poblarla con JOIN (forzando collation para evitar el 1267)
        cur.execute("""
            UPDATE planificacion.reserva_conf rc
            JOIN (
              SELECT Central, MAX(Subtipo) AS Subtipo
              FROM costos_variables.prg_energia_horaria
              WHERE Central IS NOT NULL AND Central <> ''
              GROUP BY Central
            ) prg
              ON rc.Central COLLATE utf8mb4_0900_ai_ci
               = prg.Central COLLATE utf8mb4_0900_ai_ci
            SET rc.Subtipo = prg.Subtipo
        """)
        conn.commit()

        # (opcional) ver cuántas quedaron sin subtipo
        cur.execute("""
            SELECT COUNT(*)
            FROM planificacion.reserva_conf
            WHERE Subtipo IS NULL OR Subtipo=''
        """)
        print("Filas sin Subtipo:", cur.fetchone()[0])

    finally:
        conn.close()

def set_subtipo_carbon_en_reserva_conf():
    conn = mysql.connector.connect(**MYSQL)
    try:
        cur = conn.cursor()
        placeholders = ",".join(["%s"] * len(CENTRALES_CARBON))

        # Forzamos misma collation en ambos lados para evitar 1267 (ajusta si tu DB usa otra)
        sql = f"""
            UPDATE planificacion.reserva_conf rc
            SET rc.Subtipo = 'Carbón'
            WHERE rc.Central COLLATE utf8mb4_0900_ai_ci IN (
                SELECT x.Central COLLATE utf8mb4_0900_ai_ci
                FROM (
                    SELECT %s AS Central
                    UNION ALL SELECT %s
                    UNION ALL SELECT %s
                    UNION ALL SELECT %s
                ) x
            );
        """

        cur.execute(sql, tuple(CENTRALES_CARBON))
        conn.commit()

        cur.execute(f"""
            SELECT Central, COUNT(*) AS n
            FROM planificacion.reserva_conf
            WHERE Central IN ({placeholders})
            GROUP BY Central
            ORDER BY Central;
        """, tuple(CENTRALES_CARBON))
        print("Filas afectadas por central:")
        for central, n in cur.fetchall():
            print(f"- {central}: {n}")

    finally:
        conn.close()

def col_safe(s: str) -> str:
    s = (s or "").strip().replace("\u00A0", " ")
    # quitar acentos comunes
    s = s.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnN"))
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^0-9A-Za-z_+\-]", "", s)
    if not s:
        s = "SIN_SUBTIPO"
    if s[0].isdigit():
        s = "_" + s
    return s

def build_reservas_tec_df(conf: pd.DataFrame) -> pd.DataFrame:
    """
    Devuelve DF ancho con columnas:
      Fecha, hora,
      CPF+_<Subtipo>, CPF-_<Subtipo>, CSF+_<Subtipo>, CSF-_<Subtipo>, CTF+_<Subtipo>, CTF-_<Subtipo>
    donde '+' usa SUBIDA y '-' usa BAJADA.
    """
    df = conf.copy()

    # normalización mínima
    df["TipoReserva"] = df["TipoReserva"].astype(str).str.strip()
    df["Subtipo"] = df["Subtipo"].astype(str).str.strip().replace({"": "SIN_SUBTIPO"})
    df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date
    df["hora"] = df["hora"].astype(int)
    df["SUBIDA"] = pd.to_numeric(df["SUBIDA"], errors="coerce").fillna(0.0)
    df["BAJADA"] = pd.to_numeric(df["BAJADA"], errors="coerce").fillna(0.0)

    # filtrar solo tipos que mapean
    df = df[df["TipoReserva"].isin(TIPO2PREF.keys())].copy()
    if df.empty:
        return pd.DataFrame(columns=[*KEY_COLS])

    # agregar prefijo CPF/CSF/CTF
    df["PREF"] = df["TipoReserva"].map(TIPO2PREF)

    # agrupar (sum) por Fecha,hora,PREF,Subtipo
    g = (
        df.groupby(["Fecha", "hora", "PREF", "Subtipo"], as_index=False)[["SUBIDA", "BAJADA"]]
          .sum()
    )

    # pivot SUBIDA -> columnas "PREF+_<Subtipo>"
    sub = g.pivot_table(
        index=["Fecha", "hora"],
        columns=["PREF", "Subtipo"],
        values="SUBIDA",
        aggfunc="sum",
        fill_value=0.0,
    )

    # pivot BAJADA -> columnas "PREF-_<Subtipo>"
    baj = g.pivot_table(
        index=["Fecha", "hora"],
        columns=["PREF", "Subtipo"],
        values="BAJADA",
        aggfunc="sum",
        fill_value=0.0,
    )

    # a columnas planas y "seguras"
    def flatten_cols(multi_cols, sign: str) -> list[str]:
        out = []
        for pref, subt in multi_cols:
            out.append(f"{pref}{sign}_{col_safe(subt)}")
        return out

    sub.columns = flatten_cols(sub.columns, "+")
    baj.columns = flatten_cols(baj.columns, "-")

    wide = pd.concat([sub, baj], axis=1).reset_index()

    # ordenar columnas: Fecha,hora, luego por prefijo y signo
    def sort_key(c: str):
        if c in ("Fecha", "hora"):
            return (0, c)
        # ejemplo: CPF+_Carbon
        m = re.match(r"^(CPF|CSF|CTF)([+\-])_", c)
        if not m:
            return (9, c)
        pref, sign = m.group(1), m.group(2)
        pref_order = {"CPF": 1, "CSF": 2, "CTF": 3}[pref]
        sign_order = {"+": 1, "-": 2}[sign]
        return (pref_order, sign_order, c)

    cols_sorted = sorted(wide.columns, key=sort_key)
    wide = wide[cols_sorted]

    return wide

def ensure_table(conn, df: pd.DataFrame):
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_OUT}`;")

    value_cols = [c for c in df.columns if c not in KEY_COLS]
    cols_sql = ",\n  ".join([f"{q_ident(c)} DECIMAL(18,6) NULL" for c in value_cols])

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS `{DB_OUT}`.`{T_OUT}` (
          Fecha DATE NOT NULL,
          hora  TINYINT NOT NULL,
          {cols_sql},
          PRIMARY KEY (Fecha, hora)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)

    # asegurar columnas si aparecen nuevas
    for c in value_cols:
        try:
            cur.execute(f"ALTER TABLE `{DB_OUT}`.`{T_OUT}` ADD COLUMN {q_ident(c)} DECIMAL(18,6) NULL;")
        except mysql.connector.Error:
            pass

    conn.commit()

def upsert_df(conn, df: pd.DataFrame, chunk_size: int = 5000):
    if df.empty:
        print("DF vacío: nada que subir.")
        return

    key_cols = KEY_COLS
    value_cols = [c for c in df.columns if c not in key_cols]

    cols_insert = [*key_cols, *value_cols]
    placeholders = ", ".join(["%s"] * len(cols_insert))
    updates = ", ".join([f"{q_ident(c)}=VALUES({q_ident(c)})" for c in value_cols])

    sql = f"""
        INSERT INTO `{DB_OUT}`.`{T_OUT}` ({", ".join(q_ident(c) for c in cols_insert)})
        VALUES ({placeholders})
        ON DUPLICATE KEY UPDATE
          {updates};
    """

    cur = conn.cursor()

    rows = df.itertuples(index=False, name=None)
    batch = []
    n = 0
    for r in rows:
        # r ya viene en el orden df.columns
        # convertimos a tuple con float/None para valores
        r = list(r)
        # Fecha,hora están primero por construcción
        for j, c in enumerate(df.columns):
            if c in key_cols:
                continue
            v = r[j]
            r[j] = None if pd.isna(v) else float(v)
        batch.append(tuple(r))
        if len(batch) >= chunk_size:
            cur.executemany(sql, batch)
            conn.commit()
            n += len(batch)
            batch = []

    if batch:
        cur.executemany(sql, batch)
        conn.commit()
        n += len(batch)

    print(f"OK: upsert {n} filas en {DB_OUT}.{T_OUT}")

def run_build_and_upload():
    conn = mysql.connector.connect(**MYSQL)
    try:
        # leer solo columnas necesarias
        conf = pd.read_sql(
            f"SELECT Fecha, hora, TipoReserva, SUBIDA, BAJADA, Subtipo FROM {T_SRC}",
            conn
        )

        reservas_tec_df = build_reservas_tec_df(conf)

        ensure_table(conn, reservas_tec_df)
        upsert_df(conn, reservas_tec_df)

        print("Columnas creadas:", len(reservas_tec_df.columns))
        print("Primeras 15:", list(reservas_tec_df.columns[:15]))

    finally:
        conn.close()

## Descarga Operación Real

In [30]:
options = Options()
options.add_argument("--start-maximized")

prefs = {
    "download.default_directory": str(DOWNLOAD_DIR.resolve()),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 180)

try:
    driver.get(MASHUP_URL)
    wait.until(EC.presence_of_element_located((By.XPATH, "//div[@data-testid='collapsed-title-Año']")))

    try:
        wait.until(EC.invisibility_of_element_located((By.ID, "inline-overlay")))
    except Exception:
        pass

    anio = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@data-testid='collapsed-title-Año']")))
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", anio)
    driver.execute_script("arguments[0].click();", anio)
    tm.sleep(1)
    seleccionar_valores(driver, wait, valores=ANIOS, seleccionar_todos=SELECCIONAR_TODOS_ANIOS)
    confirmar_seleccion(driver, wait, "Año")
    tm.sleep(1)

    mes = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@data-testid='collapsed-title-Mes']")))
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", mes)
    driver.execute_script("arguments[0].click();", mes)
    tm.sleep(1)
    seleccionar_valores(driver, wait, valores=MESES_SELECCION, seleccionar_todos=SELECCIONAR_TODOS_MESES)
    confirmar_seleccion(driver, wait, "Mes")
    tm.sleep(1)

    click_descargar(driver, wait)
    descargado = esperar_descarga_finalizada(DOWNLOAD_DIR, timeout=1000)
    final_path = mover_y_renombrar(descargado, DOWNLOAD_DIR, FINAL_NAME)

    print("Descargado y guardado en:", final_path)

finally:
    driver.quit()


Descargado y guardado en: /Users/valentinauretadelafuente/Desktop/Operacion Real/Datos Operación/Generacion_real_2026_Mar.csv


## Descarga barras 

In [31]:
from pathlib import Path

DOWNLOADS = Path("/Users/valentinauretadelafuente/Downloads")
XLSX_NAME = "9cfc559d-1014-4c6c-9cf1-9a7ab25a2fe6.xlsx"  
XLSX_PATH = DOWNLOADS / XLSX_NAME

if not XLSX_PATH.exists():
    raise FileNotFoundError(f"No existe: {XLSX_PATH}")

print("Usando:", XLSX_PATH)

Usando: /Users/valentinauretadelafuente/Downloads/9cfc559d-1014-4c6c-9cf1-9a7ab25a2fe6.xlsx


In [32]:
import pandas as pd

CSV_PATH = XLSX_PATH.with_suffix(".csv")

SHEET_NAME = 0  # 0 = primera hoja

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET_NAME)
df.to_csv(CSV_PATH, sep=";", index=False, encoding="utf-8", lineterminator="\n")

print("CSV generado:", CSV_PATH)
print("Columnas:", list(df.columns))

conn = mysql.connector.connect(
    host=MYSQL_HOST, port=MYSQL_PORT,
    user=MYSQL_USER, password=MYSQL_PASSWORD,
    database=MYSQL_DB,
    allow_local_infile=True,
)
try:
    cur = conn.cursor()

    # Asegurar llave única (si ya existe, se ignora)
    try:
        cur.execute("ALTER TABLE datos_operacion_real.barras ADD UNIQUE KEY uq_fecha_hora (Fecha, Hora);")
    except mysql.connector.Error:
        pass

    sql = """
    LOAD DATA LOCAL INFILE %s
    IGNORE
    INTO TABLE datos_operacion_real.barras
    CHARACTER SET utf8mb4
    FIELDS TERMINATED BY ';'
    LINES TERMINATED BY '\\n'
    IGNORE 1 LINES
    (
      @Fecha,
      @ignorar1,
      @ignorar2,
      @ignorar3,
      @Hora_txt,
      @ignorar4,
      @A_JAHUEL,
      @CHARRUA,
      @CRUCERO,
      @D_ALMAGRO,
      @MAITENCILLO,
      @P_MONTT,
      @QUILLOTA
    )
    SET
      Fecha = STR_TO_DATE(@Fecha, '%Y-%m-%d'),
      Hora  = CAST(@Hora_txt AS UNSIGNED),

      A_JAHUEL     = NULLIF(REPLACE(@A_JAHUEL,     ',', '.'), ''),
      CHARRUA      = NULLIF(REPLACE(@CHARRUA,      ',', '.'), ''),
      CRUCERO      = NULLIF(REPLACE(@CRUCERO,      ',', '.'), ''),
      D_ALMAGRO    = NULLIF(REPLACE(@D_ALMAGRO,    ',', '.'), ''),
      MAITENCILLO  = NULLIF(REPLACE(@MAITENCILLO,  ',', '.'), ''),
      P_MONTT      = NULLIF(REPLACE(@P_MONTT,      ',', '.'), ''),
      QUILLOTA     = NULLIF(REPLACE(@QUILLOTA,     ',', '.'), '');
    """

    cur.execute(sql, (str(CSV_PATH),))
    conn.commit()
    print("LOAD DATA ejecutado. Rowcount (orientativo):", cur.rowcount)

finally:
    conn.close()


CSV generado: /Users/valentinauretadelafuente/Downloads/9cfc559d-1014-4c6c-9cf1-9a7ab25a2fe6.csv
Columnas: ['Fecha', 'Versión', 'Último Dato Definitivo', 'Dia', 'Hora', 'Barra', 'A.JAHUEL______220', 'CHARRUA_______220', 'CRUCERO_______220', 'D.ALMAGRO_____220', 'MAITENCILLO___220', 'P.MONTT_______220', 'QUILLOTA______220']
LOAD DATA ejecutado. Rowcount (orientativo): 240


## Descarga Programación


In [34]:
_RE_YYYYMMDD = re.compile(r"(\d{8})")

def parse_yyyymmdd_from_filename(name: str) -> Optional[str]:
    m = _RE_YYYYMMDD.search(name)
    return m.group(1) if m else None

def yyyymmdd_to_date(s: str) -> date:
    return date(int(s[0:4]), int(s[4:6]), int(s[6:8]))

def procesar_programa_zip(zip_path: Path, programas_prg: Path, programas_po: Path, tmp: Path) -> Tuple[Optional[Path], Optional[Path]]:
    """
    Extrae PRG/PO desde PROGRAMA*.zip y los mueve a Programas/PRG y Programas/PO.
    """
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmp)

    # Buscar fecha en nombre del zip
    yyyymmdd = parse_yyyymmdd_from_filename(zip_path.name)
    if not yyyymmdd:
        return None, None
    f = yyyymmdd_to_date(yyyymmdd)
    yymmdd = f.strftime("%y%m%d")

    salida_prg = programas_prg / f"PRG{yymmdd}.xlsx"
    salida_po = programas_po / f"PO{yymmdd}.xlsx"

    # Si ya existen, no sobrescribir (evita duplicados)
    if salida_prg.exists() and salida_po.exists():
        return salida_prg, salida_po

    prg_src = tmp / f"PRG{yymmdd}.xlsx"
    po_src = tmp / f"PO{yymmdd}.xlsx"

    if not prg_src.exists():
        cands = list(tmp.glob(f"PRG{yymmdd}*.xlsx"))
        if cands:
            prg_src = max(cands, key=lambda p: p.stat().st_mtime)

    if not po_src.exists():
        cands = list(tmp.glob(f"PO{yymmdd}*.xlsx"))
        if cands:
            po_src = max(cands, key=lambda p: p.stat().st_mtime)

    prg_out = None
    po_out = None

    if prg_src.exists() and not salida_prg.exists():
        shutil.move(str(prg_src), str(salida_prg))
    if salida_prg.exists():
        prg_out = salida_prg

    if po_src.exists() and not salida_po.exists():
        shutil.move(str(po_src), str(salida_po))
    if salida_po.exists():
        po_out = salida_po

    # limpiar tmp extraído
    for p in tmp.iterdir():
        if p.is_file() and p.suffix.lower() in {".xlsx", ".xlsm"}:
            try:
                p.unlink()
            except Exception:
                pass

    return prg_out, po_out

def procesar_tco_zip(zip_path: Path, tco_dir: Path, tmp: Path) -> Optional[Path]:
    """
    Extrae TCO*.xlsm desde TCO*.zip y lo mueve a TCO/TCOyyyymmdd.xlsm.
    """
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmp)

    yyyymmdd = parse_yyyymmdd_from_filename(zip_path.name)
    if not yyyymmdd:
        return None

    salida = tco_dir / f"TCO{yyyymmdd}.xlsm"
    if salida.exists():
        return salida

    # candidatos dentro del zip
    xlsm_cands = [
        tmp / f"TCO{yyyymmdd}.xlsm",
        tmp / f"TCO{yyyymmdd}-1.xlsm",
        tmp / f"TCO{yyyymmdd}-2.xlsm",
    ]
    src = next((p for p in xlsm_cands if p.exists()), None)
    if src is None:
        cands = list(tmp.glob(f"TCO{yyyymmdd}*.xlsm"))
        if not cands:
            return None
        src = max(cands, key=lambda p: p.stat().st_mtime)

    shutil.move(str(src), str(salida))

    # limpiar tmp extraído
    for p in tmp.iterdir():
        if p.is_file() and p.suffix.lower() in {".xlsx", ".xlsm"}:
            try:
                p.unlink()
            except Exception:
                pass

    return salida

def descargar_todos_tco_y_programa(
    verbose: bool = True,
    resultados_por_pagina: int = 30,
) -> Tuple[Dict[str, str], Dict[str, str], Dict[str, str]]:
    """
    Retorna:
      - programa_zips_descargados: {filename: path}
      - prg_generados: {yyyymmdd: path_prg}
      - tco_generados: {yyyymmdd: path_tco}
    """
    base_dir = Path.cwd()
    prg_dir, po_dir, tco_dir, zips_dir, tmp = asegurar_carpetas(base_dir)

    driver = make_driver(tmp)
    wait = WebDriverWait(driver, 25)

    programa_zips_descargados: Dict[str, str] = {}
    prg_generados: Dict[str, str] = {}
    tco_generados: Dict[str, str] = {}

    vistos_en_tabla: Set[str] = set()  # para saltar duplicados dentro de la tabla

    try:
        driver.get(URL_DOCS)
        wait.until(EC.presence_of_element_located((By.XPATH, "//*[contains(.,'Bases del Modelo')]")))
        wait.until(EC.presence_of_element_located((By.XPATH, "//table//tbody")))

        # 1) Cambiar resultados por página a 30
        set_resultados_por_pagina(driver, wait, resultados_por_pagina)

        # 2) Recorrer páginas y descargar
        while True:
            _try_close_common_popups(driver)
            _kill_overlays(driver)

            filas = obtener_filas_tabla(driver, wait)

            for row in filas:
                nombre = extraer_nombre_archivo_de_fila(row)
                if not nombre:
                    continue

                # Solo PROGRAMA*.zip y TCO*.zip
                if not (nombre.startswith("PROGRAMA") or nombre.startswith("TCO")):
                    continue
                if not nombre.lower().endswith(".zip"):
                    continue

                # Evitar duplicados en tabla
                if nombre in vistos_en_tabla:
                    continue
                vistos_en_tabla.add(nombre)

                yyyymmdd = parse_yyyymmdd_from_filename(nombre)
                if not yyyymmdd:
                    continue

                # Evitar guardar repetidos en carpeta:
                # - PROGRAMA -> si PRG y PO ya existen, no descargar
                # - TCO -> si TCOyyyymmdd.xlsm ya existe, no descargar
                if nombre.startswith("PROGRAMA"):
                    f = yyyymmdd_to_date(yyyymmdd)
                    yymmdd = f.strftime("%y%m%d")
                    salida_prg = prg_dir / f"PRG{yymmdd}.xlsx"
                    salida_po = po_dir / f"PO{yymmdd}.xlsx"
                    if salida_prg.exists() and salida_po.exists():
                        if verbose:
                            print(f"[SKIP] {nombre} (PRG/PO ya existen)")
                        continue

                if nombre.startswith("TCO"):
                    salida_tco = tco_dir / f"TCO{yyyymmdd}.xlsm"
                    if salida_tco.exists():
                        if verbose:
                            print(f"[SKIP] {nombre} (TCO ya existe)")
                        continue

                # Evitar redescargar el mismo zip si ya está guardado en ZIPS/
                zip_guardado = zips_dir / nombre
                if zip_guardado.exists():
                    if verbose:
                        print(f"[ZIP ] YA  {nombre}")
                    zip_path = zip_guardado
                else:
                    limpiar_tmp(tmp, [f"{nombre}", f"{nombre}.crdownload", "*.crdownload"])
                    if verbose:
                        print(f"[ZIP ] DOWN {nombre}")

                    click_descarga_en_fila(driver, row)
                    zip_path_tmp = esperar_archivo_sin_crdownload(tmp, nombre, timeout_s=300)
                    if zip_path_tmp is None:
                        if verbose:
                            print(f"[ERR] No bajó: {nombre}")
                        continue

                    # mover zip a ZIPS/ para cache y evitar repetidos
                    shutil.move(str(zip_path_tmp), str(zip_guardado))
                    zip_path = zip_guardado

                programa_zips_descargados[nombre] = str(zip_path)

                # Procesar zip
                if nombre.startswith("PROGRAMA"):
                    # extrae en tmp (limpia posibles restos)
                    limpiar_tmp(tmp, ["PRG*.xlsx", "PO*.xlsx", "*.xlsx", "*.xlsm"])
                    prg_path, po_path = procesar_programa_zip(zip_path, prg_dir, po_dir, tmp)
                    if prg_path:
                        prg_generados[yyyymmdd] = str(prg_path)
                    if verbose:
                        print(f"[PRG/PO] OK {nombre} -> {prg_path.name if prg_path else 'PRG?'} / {po_path.name if po_path else 'PO?'}")

                elif nombre.startswith("TCO"):
                    limpiar_tmp(tmp, ["TCO*.xlsm", "*.xlsm"])
                    tco_path = procesar_tco_zip(zip_path, tco_dir, tmp)
                    if tco_path:
                        tco_generados[yyyymmdd] = str(tco_path)
                        if verbose:
                            print(f"[TCO] OK {nombre} -> {tco_path.name}")
                    else:
                        if verbose:
                            print(f"[TCO] NO {nombre}")

            # 3) Siguiente página; si no hay, terminar
            if not ir_a_siguiente_pagina(driver):
                break

        return programa_zips_descargados, prg_generados, tco_generados

    finally:
        driver.quit()

if __name__ == "__main__":
    zips, prg, tco = descargar_todos_tco_y_programa(verbose=True, resultados_por_pagina=30)
    print(f"\nTotal ZIPs: {len(zips)}")
    print(f"Total PRG : {len(prg)}")
    print(f"Total TCO : {len(tco)}")

[ZIP ] DOWN PROGRAMA20260328.zip
[PRG/PO] OK PROGRAMA20260328.zip -> PRG260328.xlsx / PO260328.xlsx
[ZIP ] DOWN TCO20260328.zip
[TCO] OK TCO20260328.zip -> TCO20260328.xlsm
[ZIP ] DOWN PROGRAMA20260327.zip
[PRG/PO] OK PROGRAMA20260327.zip -> PRG260327.xlsx / PO260327.xlsx
[ZIP ] DOWN TCO20260327.zip
[TCO] OK TCO20260327.zip -> TCO20260327.xlsm
[ZIP ] DOWN PROGRAMA20260326.zip
[PRG/PO] OK PROGRAMA20260326.zip -> PRG260326.xlsx / PO260326.xlsx
[ZIP ] DOWN TCO20260326.zip
[TCO] OK TCO20260326.zip -> TCO20260326.xlsm
[ZIP ] DOWN TCO20260325.zip
[TCO] OK TCO20260325.zip -> TCO20260325.xlsm
[ZIP ] DOWN PROGRAMA20260325.zip
[PRG/PO] OK PROGRAMA20260325.zip -> PRG260325.xlsx / PO260325.xlsx
[ZIP ] DOWN PROGRAMA20260324.zip
[PRG/PO] OK PROGRAMA20260324.zip -> PRG260324.xlsx / PO260324.xlsx

Total ZIPs: 9
Total PRG : 5
Total TCO : 4


## Programas

In [40]:
conn = mysql.connector.connect(**MYSQL)
try:
    migrar_esquema(conn)
    print("OK esquema")
finally:
    conn.close()

SQL_UPSERT_ENERGIA = """
INSERT INTO prg_energia_horaria
(Fecha, Central, Categoria, Subtipo,
 Hora1, Hora2, Hora3, Hora4, Hora5, Hora6, Hora7, Hora8, Hora9, Hora10, Hora11, Hora12,
 Hora13, Hora14, Hora15, Hora16, Hora17, Hora18, Hora19, Hora20, Hora21, Hora22, Hora23, Hora24)
VALUES
(%s, %s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
  Categoria=VALUES(Categoria),
  Hora1=VALUES(Hora1), Hora2=VALUES(Hora2), Hora3=VALUES(Hora3), Hora4=VALUES(Hora4),
  Hora5=VALUES(Hora5), Hora6=VALUES(Hora6), Hora7=VALUES(Hora7), Hora8=VALUES(Hora8),
  Hora9=VALUES(Hora9), Hora10=VALUES(Hora10), Hora11=VALUES(Hora11), Hora12=VALUES(Hora12),
  Hora13=VALUES(Hora13), Hora14=VALUES(Hora14), Hora15=VALUES(Hora15), Hora16=VALUES(Hora16),
  Hora17=VALUES(Hora17), Hora18=VALUES(Hora18), Hora19=VALUES(Hora19), Hora20=VALUES(Hora20),
  Hora21=VALUES(Hora21), Hora22=VALUES(Hora22), Hora23=VALUES(Hora23), Hora24=VALUES(Hora24);
"""

SQL_UPSERT_CM = """
INSERT INTO costos_marginales
(Fecha, Barra, Codigo,
 Hora1, Hora2, Hora3, Hora4, Hora5, Hora6, Hora7, Hora8, Hora9, Hora10, Hora11, Hora12,
 Hora13, Hora14, Hora15, Hora16, Hora17, Hora18, Hora19, Hora20, Hora21, Hora22, Hora23, Hora24)
VALUES
(%s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
  Codigo=VALUES(Codigo),
  Hora1=VALUES(Hora1), Hora2=VALUES(Hora2), Hora3=VALUES(Hora3), Hora4=VALUES(Hora4),
  Hora5=VALUES(Hora5), Hora6=VALUES(Hora6), Hora7=VALUES(Hora7), Hora8=VALUES(Hora8),
  Hora9=VALUES(Hora9), Hora10=VALUES(Hora10), Hora11=VALUES(Hora11), Hora12=VALUES(Hora12),
  Hora13=VALUES(Hora13), Hora14=VALUES(Hora14), Hora15=VALUES(Hora15), Hora16=VALUES(Hora16),
  Hora17=VALUES(Hora17), Hora18=VALUES(Hora18), Hora19=VALUES(Hora19), Hora20=VALUES(Hora20),
  Hora21=VALUES(Hora21), Hora22=VALUES(Hora22), Hora23=VALUES(Hora23), Hora24=VALUES(Hora24);
"""

OK esquema


In [41]:
def upsert_energia_dict(cur, fecha: date, dict_energia: Dict[str, Any]):
    for central, (_codigo, categoria, serie) in dict_energia.items():
        categoria = str(categoria)
        subtipo = None

        if categoria.strip().lower() in ("térmicas", "termicas"):
            subtipo = subtipo_termica_por_nombre(central)

        vals = serie_a_24(serie)
        cur.execute(SQL_UPSERT_ENERGIA, (fecha, central, categoria, subtipo, *vals))


def upsert_costos_marginales(cur, fecha: date, dict_costos: Dict[str, Any]):
    for barra, (codigo, serie) in dict_costos.items():
        vals = serie_a_24(serie)
        cur.execute(SQL_UPSERT_CM, (fecha, barra, codigo, *vals))


def upsert_sistemas_almacenamiento(cur, fecha: date, dict_sistemas: Dict[str, Any]):
    """
    Sube la tabla 'Sistemas de Almacenamiento' a energía con:
      Categoria = 'Sistemas de Almacenamiento'
      Subtipo   = NULL
    """
    for nombre, (_codigo, serie) in dict_sistemas.items():
        vals = serie_a_24(serie)
        cur.execute(
            SQL_UPSERT_ENERGIA,
            (fecha, str(nombre), "Sistemas de Almacenamiento", None, *vals)
        )


def upsert_vertimientos(cur, fecha: date, dict_reduccion: Dict[str, Any]):
    """
    Sube 'Reducción de Renovable Estimada' a energía con:
      Categoria = 'Vertimientos'
      Subtipo   = 'Vertimientos'

    Si quieres solo la fila Total, se filtra aquí.
    """
    for nombre, serie in dict_reduccion.items():
        if str(nombre).strip().lower() != "total":
            continue

        vals = serie_a_24(serie)
        cur.execute(
            SQL_UPSERT_ENERGIA,
            (fecha, "Total Vertimiento", "Ver", "Vertimientos", *vals)
        )


def subir_prg_completo(prg_path, fecha: date):
    dict_energia, dict_costos, dict_sistemas, dict_reduccion = leer_prg(prg_path)

    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT,
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        database=MYSQL_DB,
    )
    try:
        cur = conn.cursor()

        upsert_energia_dict(cur, fecha, dict_energia)
        upsert_costos_marginales(cur, fecha, dict_costos)
        upsert_sistemas_almacenamiento(cur, fecha, dict_sistemas)
        upsert_vertimientos(cur, fecha, dict_reduccion)

        conn.commit()
        print("OK:", fecha, prg_path)

    finally:
        conn.close()


def subir_prg_rango(folder_prg: str | Path, fecha_inicio: date, fecha_fin: date):
    folder_prg = Path(folder_prg)

    f = fecha_inicio
    while f <= fecha_fin:
        prg_name = f"PRG{f.strftime('%y%m%d')}.xlsx"
        prg_path = folder_prg / prg_name

        try:
            subir_prg_completo(prg_path, f)
            print("[OK]", f, prg_path.name)
        except FileNotFoundError:
            print("[NO FILE]", f, prg_path.name)
        except Exception as e:
            print("[ERROR]", f, prg_path.name, "->", e)

        f += timedelta(days=1)
def corregir_subtipo_por_categoria_eliminando_conflictos():
    conn = mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
    )
    try:
        cur = conn.cursor()

        # Mapeo Categoria -> Subtipo (ajusta strings exactamente a como están en tu tabla)
        mappings = [
            ("Hidroelectricas de Pasada", "Pasada"),
            ("Embalses y Reguladas",      "Embalse"),
            ("Eólicas",                   "Eólica"),
            ("Solares",                   "Solar"),
            ("Centrales de concentración solar","CS")
        ]

        conn.start_transaction()

        total_deleted = 0
        total_updated = 0

        # 1) Borrar filas que causarían duplicado al actualizar
        for categoria, subtipo_obj in mappings:
            # Borra SOLO las filas con Subtipo NULL/'' que chocarían con otra fila existente
            delete_sql = """
            DELETE a
            FROM costos_variables.prg_energia_horaria a
            JOIN costos_variables.prg_energia_horaria b
              ON b.Fecha = a.Fecha
             AND b.Central = a.Central
             AND b.Subtipo = %s
            WHERE a.Categoria = %s
              AND (a.Subtipo IS NULL OR a.Subtipo = '')
              AND a.id <> b.id;
            """
            cur.execute(delete_sql, (subtipo_obj, categoria))
            total_deleted += cur.rowcount

        # 2) Actualizar Subtipo
        update_sql = """
        UPDATE costos_variables.prg_energia_horaria
        SET Subtipo = CASE
            WHEN Categoria = 'Hidroelectricas de Pasada' THEN 'Pasada'
            WHEN Categoria = 'Embalses y Reguladas'      THEN 'Embalse'
            WHEN Categoria = 'Eólicas'                   THEN 'Eólica'
            WHEN Categoria = 'Solares'                   THEN 'Solar'
            WHEN Categoria = 'Centrales de concentración solar' THEN 'CS'
            ELSE Subtipo
        END
        WHERE Categoria IN ('Hidroelectricas de Pasada', 'Embalses y Reguladas', 'Eólicas', 'Solares','Centrales de concentración solar')
          AND (Subtipo IS NULL OR Subtipo = '');
        """
        cur.execute(update_sql)
        total_updated = cur.rowcount

        conn.commit()
        print("Filas borradas por conflicto de llave:", total_deleted)
        print("Filas actualizadas:", total_updated)

    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


### Uso

In [7]:
ini = date(2025, 1, 1)
fin = date(2026, 3, 16)

In [42]:
# ------- PRG -------
subir_prg_rango("Programas/PRG", ini, fin)
corregir_subtipo_por_categoria_eliminando_conflictos()

OK: 2026-03-16 Programas/PRG/PRG260316.xlsx
[OK] 2026-03-16 PRG260316.xlsx
OK: 2026-03-17 Programas/PRG/PRG260317.xlsx
[OK] 2026-03-17 PRG260317.xlsx
OK: 2026-03-18 Programas/PRG/PRG260318.xlsx
[OK] 2026-03-18 PRG260318.xlsx
OK: 2026-03-19 Programas/PRG/PRG260319.xlsx
[OK] 2026-03-19 PRG260319.xlsx
OK: 2026-03-20 Programas/PRG/PRG260320.xlsx
[OK] 2026-03-20 PRG260320.xlsx
OK: 2026-03-21 Programas/PRG/PRG260321.xlsx
[OK] 2026-03-21 PRG260321.xlsx
OK: 2026-03-22 Programas/PRG/PRG260322.xlsx
[OK] 2026-03-22 PRG260322.xlsx
OK: 2026-03-23 Programas/PRG/PRG260323.xlsx
[OK] 2026-03-23 PRG260323.xlsx
OK: 2026-03-24 Programas/PRG/PRG260324.xlsx
[OK] 2026-03-24 PRG260324.xlsx
OK: 2026-03-25 Programas/PRG/PRG260325.xlsx
[OK] 2026-03-25 PRG260325.xlsx
OK: 2026-03-26 Programas/PRG/PRG260326.xlsx
[OK] 2026-03-26 PRG260326.xlsx
OK: 2026-03-27 Programas/PRG/PRG260327.xlsx
[OK] 2026-03-27 PRG260327.xlsx
OK: 2026-03-28 Programas/PRG/PRG260328.xlsx
[OK] 2026-03-28 PRG260328.xlsx
Filas borradas por confli

In [43]:
# ------- Barras programa -------
upsert_barras_from_wide_hours(ini, fin)

Filas en planificacion.barras (rango): 312


In [45]:
# ------- PO -------
po_dir = Path("Programas/PO")
conn = mysql.connector.connect(**MYSQL)
f = ini
while f <= fin:
    po_path = po_dir / f"PO{f.strftime('%y%m%d')}.xlsx"
    if not po_path.exists():
        print(f"[NO FILE] {f} -> {po_path.name}")
        f += timedelta(days=1)
        continue
    try:
        t0 = tm.time()
        po_rows = leer_po_fijo(po_path)
        t1 = tm.time()
        n = upsert_po_termicas(conn, f, po_rows)
        t2 = tm.time()
        print(
            f"[OK] {f} -> {po_path.name} ({len(po_rows)} filas) | "
            f"leer: {t1 - t0:.2f}s | db: {t2 - t1:.2f}s | total: {t2 - t0:.2f}s"
        )
    except Exception as e:
        print(f"[ERROR] {f} -> {po_path.name}: {e}")
    f += timedelta(days=1)

[OK] 2026-03-16 -> PO260316.xlsx (692 filas) | leer: 71.53s | db: 0.13s | total: 71.66s
[OK] 2026-03-17 -> PO260317.xlsx (692 filas) | leer: 71.65s | db: 0.04s | total: 71.69s
[OK] 2026-03-18 -> PO260318.xlsx (692 filas) | leer: 71.51s | db: 0.04s | total: 71.55s
[OK] 2026-03-19 -> PO260319.xlsx (692 filas) | leer: 71.72s | db: 0.04s | total: 71.76s
[OK] 2026-03-20 -> PO260320.xlsx (692 filas) | leer: 71.90s | db: 0.04s | total: 71.94s
[OK] 2026-03-21 -> PO260321.xlsx (692 filas) | leer: 336.29s | db: 0.03s | total: 336.33s
[OK] 2026-03-22 -> PO260322.xlsx (692 filas) | leer: 78.03s | db: 0.04s | total: 78.07s
[OK] 2026-03-23 -> PO260323.xlsx (692 filas) | leer: 71.61s | db: 0.15s | total: 71.76s
[OK] 2026-03-24 -> PO260324.xlsx (692 filas) | leer: 214.75s | db: 0.04s | total: 214.79s
[OK] 2026-03-25 -> PO260325.xlsx (692 filas) | leer: 71.59s | db: 0.04s | total: 71.63s
[OK] 2026-03-26 -> PO260326.xlsx (692 filas) | leer: 71.59s | db: 0.03s | total: 71.62s
[OK] 2026-03-27 -> PO260327.

In [46]:
# ------- TCO -------
tco_dir = Path("TCO")
f = ini
while f <= fin:
    base = f"TCO{f.strftime('%Y%m%d')}"
    candidatos = [
        tco_dir / f"{base}.xlsm",
        tco_dir / f"{base}-1.xlsm",
        tco_dir / f"{base}-2.xlsm",
    ]
    tco_path = next((p for p in candidatos if p.exists()), None)
    if not tco_path:
        print(f"[NO FILE] {f} -> {base}.xlsm (-1/-2)")
        f += timedelta(days=1)
        continue
    try:
        t0 = tm.time()
        rows = leer_tco_cv_fijo_rapido(tco_path, sheet_name="CV", header_row=20)
        t1 = tm.time()

        upsert_tco(conn, f, rows)  # commitea 1 vez por archivo
        t2 = tm.time()

        print(
            f"[OK] {f} -> {tco_path.name} ({len(rows)} filas) | "
            f"leer: {t1 - t0:.2f}s | db: {t2 - t1:.2f}s | total: {t2 - t0:.2f}s"
        )
    except Exception as e:
        print(f"[ERROR] {f} -> {tco_path.name}: {e}")
    f += timedelta(days=1)
actualizar_subtipos_tco()

[OK] 2026-03-16 -> TCO20260316.xlsm (814 filas) | leer: 0.22s | db: 0.08s | total: 0.30s
[OK] 2026-03-17 -> TCO20260317.xlsm (814 filas) | leer: 0.26s | db: 0.05s | total: 0.32s
[OK] 2026-03-18 -> TCO20260318.xlsm (814 filas) | leer: 0.23s | db: 0.06s | total: 0.29s
[OK] 2026-03-19 -> TCO20260319.xlsm (814 filas) | leer: 0.21s | db: 0.05s | total: 0.26s
[OK] 2026-03-20 -> TCO20260320.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-21 -> TCO20260321.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-22 -> TCO20260322.xlsm (814 filas) | leer: 0.26s | db: 0.06s | total: 0.31s
[OK] 2026-03-23 -> TCO20260323.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-24 -> TCO20260324.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.26s
[OK] 2026-03-25 -> TCO20260325.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-26 -> TCO20260326.xlsm (814 filas) | leer: 0.27s | db: 0.06s | total: 0.33s
[OK] 2026-03-27 -> TC

In [48]:
import mysql.connector

MYSQL = dict(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="rootvale123",
)

TABLAS = [
    "costos_variables.tco",
    "costos_variables.po",
]

def unificar_fuel_oil_en_tablas():
    conn = mysql.connector.connect(**MYSQL)
    try:
        cur = conn.cursor()

        for tabla in TABLAS:
            print(f"\n--- {tabla} ---")

            # Ver estado antes
            cur.execute(f"""
                SELECT Subtipo, COUNT(*)
                FROM {tabla}
                WHERE Subtipo IN ('Fuel Oil', 'Fuil Oil')
                GROUP BY Subtipo
                ORDER BY Subtipo
            """)
            antes = cur.fetchall()

            print("Antes:")
            if antes:
                for subtipo, n in antes:
                    print(f"- {subtipo}: {n}")
            else:
                print("- Sin filas con Fuel Oil / Fuil Oil")

            # Actualizar
            cur.execute(f"""
                UPDATE {tabla}
                SET Subtipo = 'Fuel Oil'
                WHERE Subtipo = 'Fuil Oil'
            """)
            conn.commit()

            print(f"Filas actualizadas: {cur.rowcount}")

            # Ver estado después
            cur.execute(f"""
                SELECT Subtipo, COUNT(*)
                FROM {tabla}
                WHERE Subtipo = 'Fuel Oil'
                GROUP BY Subtipo
            """)
            despues = cur.fetchall()

            print("Después:")
            if despues:
                for subtipo, n in despues:
                    print(f"- {subtipo}: {n}")
            else:
                print("- No quedó ninguna fila con Fuel Oil")

    finally:
        conn.close()

unificar_fuel_oil_en_tablas()


--- costos_variables.tco ---
Antes:
- Fuel Oil: 5222
- Fuil Oil: 460
Filas actualizadas: 460
Después:
- Fuel Oil: 5682

--- costos_variables.po ---
Antes:
- Fuel Oil: 5186
- Fuil Oil: 202
Filas actualizadas: 202
Después:
- Fuel Oil: 5388


In [47]:
# ------- Reservas -------
upsert_reserva_conf_rango(
     fecha_inicio=ini,
     fecha_fin=fin,
     carpeta_prg="Programas/PRG",
     patron="PRG{yy}{mm}{dd}.xlsx"
 )
add_and_fill_subtipo()
set_subtipo_carbon_en_reserva_conf()
run_build_and_upload()

OK: upsert 36456 filas en planificacion.reserva_conf (PRG260316.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260317.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260318.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260319.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260320.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260321.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260322.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260323.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260324.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260325.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260326.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260327.xlsx)
OK: upsert 36480 filas en planificacion.reserva_conf (PRG260328.xlsx)
Resumen: OK=13, NO_FILE=0, ERROR=0
Filas sin Subtipo: 21288
Filas afectadas por central:
-

/var/folders/b1/j2t3b3w12dl60tdztfsl3b0w0000gn/T/ipykernel_72798/2527679455.py:1394: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  conf = pd.read_sql(


OK: upsert 10752 filas en planificacion.reservas_tec
Columnas creadas: 64
Primeras 15: ['Fecha', 'hora', 'CPF+_Carbon', 'CPF+_Diesel', 'CPF+_Embalse', 'CPF+_Eolica', 'CPF+_GNA', 'CPF+_GNL', 'CPF+_GNL_INF', 'CPF+_None', 'CPF+_Solar', 'CPF-_Carbon', 'CPF-_Diesel', 'CPF-_Embalse', 'CPF-_Eolica']
